<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_pinn_plate_switched_on_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 — The Plate Switched On

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L8.2 · Transient Heat**

The plate of Ex_08.1 is cold, at the coolant's temperature, and its power is
switched on. How fast does it warm, and when is it steady? It is the heat
equation: the steady problem with one more term, the heat the plate stores.
You solve it three ways - a finite-element reference, finite differences, and
a physics-informed network with the start and the channel built in - and
compare them on accuracy and on time. Then the same network finds the plate's
diffusivity from four thermocouples.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and finite differences on a grid for an agreed 0.20 K | — |
| **5** | a PINN with the start and the channel built in, and the comparison | the trial function, the residual |
| **6** | the inverse problem: the diffusivity from four thermocouples | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About eight minutes on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex08.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

The aluminium plate of Ex_08.1 - 100 × 100 mm, a coolant channel at its centre,
insulated outer edges; the channel is the hole of the lecture's slides - sits at the coolant's 40 °C. At $t = 0$ its electronics
are switched on and generate 1 MW/m³. What is the temperature at every point
over the next 80 seconds, and how long before the plate is as hot as it gets?

## 2 · The data

| | |
|---|---|
| plate | 100 × 100 mm, aluminium alloy 6061, $k$ = 167 W/(m K), $\rho c_p$ = 2.43 MJ/(m³ K) |
| diffusivity | $\alpha = k/(\rho c_p)$ = 6.87e-05 m²/s |
| heat generated | $Q$ = 1 MW/m³, evenly, from $t = 0$ |
| cooling channel | an ellipse of 36 × 22 mm at the centre, its wall held at 40 °C |
| outer edges, start | insulated; the whole plate at 40 °C at $t = 0$ |
| window | 0 to 80 s |

## 3 · The physics

The heat that is generated and not conducted away is **stored**, and the
temperature rises (L8.2's first section):

$$\rho c_p\,T_t = k\,(T_{xx} + T_{yy}) + Q$$

with $T$ = 40 °C on the channel and at the start, and no flux through the
edges. The equation is **parabolic**: it has a direction in time, needs **one**
initial field, and smooths as it goes. Heat crosses the plate in a time
$L^2/\alpha$ = 145.51 s, which says what window to look at before anything is
solved.

**Scaled units.** Lengths over $L$, the rise above the coolant over
$\Delta T = QL^2/k$ = 59.88 K, and time over the 80 s window, $\tau = t/80$:

$$\theta_\tau = C\,(\theta_{xx} + \theta_{yy} + 1), \qquad C = \frac{\alpha\cdot 80\ \text{s}}{L^2} = 0.55$$

with $\theta = 0$ on the channel and at $\tau = 0$. Everything below is solved
in these units and printed in kelvin and seconds. The cell below prints the
data and draws the plate, with the four thermocouples section 6 uses.

In [ ]:
pb.describe_problem()
L_mm = pb.L_PLATE * 1e3
t = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot([0, L_mm, L_mm, 0, 0], [0, 0, L_mm, L_mm, 0], "--", color="tab:blue", label="insulated edges")
ax.plot(L_mm * (pb.HOLE["xc"] + pb.HOLE["a"] * np.cos(t)), L_mm * (pb.HOLE["yc"] + pb.HOLE["b"] * np.sin(t)),
        color="tab:orange", label="channel wall, 40 °C")
ax.plot(pb.SENSORS[:, 0] * L_mm, pb.SENSORS[:, 1] * L_mm, "k^", ms=7, label="thermocouples")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("y  [mm]")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
plt.tight_layout(); plt.show()

**What you should see.** The data of section 2, a diffusivity of 6.87e-05 m²/s,
145.51 s for heat to cross the plate and a scaled diffusivity of 0.55; and the
plate with its channel and four thermocouples.

---

## 4 · The reference, and finite differences

**The reference.** As in Ex_08.1 there is no exact solution, so the methods are
scored against a **converged numerical solution**: finite elements on the mesh
fitted to the channel, and Crank-Nicolson in time. `pb.reference(spacing,
steps)` builds it. The cell below halves the node spacing and the time step
together, twice, and prints how far each solution is from the next.

In [ ]:
g = np.linspace(0, 1, 101); GX, GY = np.meshgrid(g, g)       # the points the references are compared on
plate = pb.ellipse_phi(np.c_[GX.ravel(), GY.ravel()]).reshape(GX.shape) > 0   # True in the plate
fields = {}
for n, steps in ((50, 80), (100, 160), (200, 320)):
    t0 = time.perf_counter(); r = pb.reference(1 / n, steps); sec = time.perf_counter() - t0
    fields[n] = (r, np.array([pb.kelvin(pb.reference_at(r, GX[plate], GY[plate], tau)) for tau in r["tau"]]))
    print(f"  spacing L/{n}, {steps} steps: {len(r['P']):6d} nodes, hottest point at 80 s "
          f"{pb.T_COOLANT + pb.kelvin(r['theta'][-1].max()):.2f} °C, {sec:.2f} s")
for a, b in ((50, 100), (100, 200)):
    print(f"  L/{a} against L/{b}: worst difference over the window {np.nanmax(np.abs(fields[a][1] - fields[b][1])):.2e} K")
ref = fields[200][0]                                          # the reference from here on
TAUS = ref["tau"]                                             # 41 instants, 2 s apart
hot = pb.kelvin(ref["theta"].max(axis=1))                     # the hottest point's rise at each instant
print(f"  the hottest point reaches {hot[-1]:.2f} K at 80 s; half of that after {np.interp(hot[-1] / 2, hot, TAUS) * pb.T_END:.1f} s")

**What you should see.** The hottest point at 49.42 °C after 80 s - 9.42 K
above the coolant, against 9.56 K in the steady state of Ex_08.1, so the plate
is within 2 % of steady - and half of that rise after about 13 s. The
difference between successive solutions falls with every halving, to 9.44e-03 K
between L/100 and L/200: **more than ten times better than the
accuracy agreed below**, so L/200 with 320 steps is the reference.

**Finite differences.** The square grid of Ex_08.1, with the channel as a
staircase, and Crank-Nicolson in time: one sparse solve per step, the matrix
factorised once. **The agreed accuracy is 0.20 K** over the whole window; the
cell below finds the coarsest grid that meets it.

In [ ]:
def fdm_plate(n, steps, keep=40):
    # n x n nodes, `steps` time steps; returns the nodes and the scaled rise at the 41 stored instants
    h = 1.0 / (n - 1)
    g = np.linspace(0, 1, n); X, Y = np.meshgrid(g, g)
    D = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n)).tolil()
    D[0, 1] = 2.0; D[-1, -2] = 2.0                             # a mirrored node: zero gradient across the edge
    D = D.tocsr() / h ** 2
    A = pb.C_SCALED * (sp.kron(sp.identity(n), D) + sp.kron(D, sp.identity(n))).tocsr()   # C times the Laplacian
    free = np.where(pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).ravel() > 0)[0]           # nodes in the plate
    A = A[free][:, free]; I = sp.identity(len(free)); dt = 1.0 / steps
    lu = spla.splu((I - 0.5 * dt * A).tocsc()); B = (I + 0.5 * dt * A).tocsr()             # Crank-Nicolson
    u = np.zeros(len(free)); out = [np.zeros(n * n)]
    for k in range(1, steps + 1):
        u = lu.solve(B @ u + dt * pb.C_SCALED)                 # one step: storage, conduction, source
        if k % (steps // keep) == 0:
            full = np.zeros(n * n); full[free] = u; out.append(full)
    return X, Y, np.array(out).reshape(-1, n, n)

def error_over_time(X, Y, U):
    # worst error against the reference at each stored instant, K, on the nodes in the plate
    m = pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape) > 0
    return np.array([np.nanmax(np.abs(pb.kelvin(U[k][m]) - pb.kelvin(pb.reference_at(ref, X[m], Y[m], tau))))
                     for k, tau in enumerate(TAUS)])

TARGET = 0.20                                                 # K: the agreed accuracy
for n, steps in ((81, 80), (161, 160), (321, 320)):
    t0 = time.perf_counter(); X, Y, U = fdm_plate(n, steps); sec = time.perf_counter() - t0
    err = error_over_time(X, Y, U).max()
    print(f"  {n:3d} x {n} nodes, {steps} steps: worst error {err:.2e} K, one run {sec:.2f} s"
          + ("   <- meets 0.20 K" if err <= TARGET else ""))
N_FDM, STEPS = 321, 320                                       # the coarsest grid that meets it

**What you should see.** The error falling slowly, as on the staircase of
Ex_08.1 - 4.62e-01, 3.15e-01 and 1.25e-01 K - and the 321 × 321 grid with 320
steps the first under 0.20 K, the whole window in five to fifteen seconds.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(x, y, \tau)$ and returns the scaled rise. It is four
layers of 32 tanh neurons, trained on 4000 collocation points spread over the
plate and the window - more of them early, where the field changes fastest.

**The start and the channel wall are built in.** The trial function

$$\hat\theta(x, y, \tau) = S\;\tau\;\phi(x, y)\;\mathcal{N}(x, y, \tau)$$

is exactly zero at $\tau = 0$ and exactly zero on the channel wall, whatever
the network does: one factor for each condition. That is the first of L8.2's
remedies for **causality** - the start cannot be fitted last if it cannot be
wrong. $S$ = 0.01 keeps the network's output of order one, as in Ex_08.1.

**The insulated edges stay a loss term**, with the weight $w_N$ = 10.

**TODO 1** is the trial function; **TODO 2** the residual. The equation is
divided by $C$, so that its source stays of size one:
$\hat\theta_\tau / C - (\hat\theta_{xx} + \hat\theta_{yy} + 1)$.

In [ ]:
SCALE = 0.01                                                  # the size of the answer over the size of phi
class Plate(torch.nn.Module):                                 # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=32, n_layers=4)       # dense tanh network: x, y, tau in
    def forward(self, p):
        tau = p[:, 2:3]
        return SCALE * tau * pb.ellipse_phi(p) * self.net(p)  # the trial function: zero at the start and on the wall

In [ ]:
def residual(model, p, c):
    th = model(p)                                             # the network's scaled rise
    th_tau = grad(th, p)[:, 2:3]                              # the time derivative by autograd; column 2 is tau
    return th_tau / c - (d2(th, p, 0) + d2(th, p, 1) + 1.0)   # the heat equation, divided by c

In [ ]:
W_N = 10.0                                                    # the weight of the insulated edges
pts = to_tensor(pb.sample_spacetime(4000), requires_grad=True)   # collocation points in space and time, differentiable
edges = [(to_tensor(q, requires_grad=True), axis) for q, axis in pb.sample_edges_spacetime(200)]   # 200 on each edge

def edge_loss(model):
    # the gradient along the normal, squared, averaged over the four edges
    return sum(grad(model(q), q)[:, axis:axis + 1].pow(2).mean() for q, axis in edges) / 4

model = Plate().to(DEVICE)                                    # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, pts, pb.C_SCALED).pow(2).mean() + W_N * edge_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
t0 = time.perf_counter(); X, Y, U = fdm_plate(N_FDM, STEPS); t_fdm = time.perf_counter() - t0   # finite differences
m = pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape) > 0
Pn = to_tensor(np.c_[X[m], Y[m]])                             # the same nodes for the network
def pinn_at(tau, net=None):
    with torch.no_grad():  # no gradient tracking: evaluation only
        col = torch.full((len(Pn), 1), float(tau), device=Pn.device, dtype=Pn.dtype)   # the time column, same device
        return to_numpy((net or model)(torch.cat([Pn, col], 1))).ravel()
P_all = np.array([pinn_at(tau) for tau in TAUS])              # the network at the 41 instants
t_pin = middle(lambda: [pinn_at(tau) for tau in TAUS], reps=3)
e_fdm = error_over_time(X, Y, U)
R_all = np.array([pb.reference_at(ref, X[m], Y[m], tau) for tau in TAUS])
e_pin = np.nanmax(np.abs(pb.kelvin(P_all) - pb.kelvin(R_all)), axis=1)

print(f"{'':30s}{'hottest at 80 s':>17s}{'worst error':>13s}{'whole window':>14s}{'training':>10s}")
print(f"  {'reference, fitted L/200':28s}{pb.T_COOLANT + hot[-1]:14.2f} °C{'-':>13s}{'-':>14s}{'-':>10s}")
print(f"  {'finite differences, 321 x 321':28s}{pb.T_COOLANT + pb.kelvin(U[-1].max()):13.2f} °C{e_fdm.max():11.2e} K{t_fdm:12.2f} s{'-':>10s}")
print(f"  {'PINN, 4 x 32':28s}{pb.T_COOLANT + pb.kelvin(P_all[-1].max()):14.2f} °C{e_pin.max():11.2e} K{t_pin:12.2f} s{train_time:8.0f} s")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(TAUS * pb.T_END, pb.T_COOLANT + hot, "k", lw=2.5, label="reference")
axes[0].plot(TAUS * pb.T_END, pb.T_COOLANT + pb.kelvin(U.reshape(len(TAUS), -1).max(axis=1)), "--", label="finite differences")
axes[0].plot(TAUS * pb.T_END, pb.T_COOLANT + pb.kelvin(P_all.max(axis=1)), ":", lw=2.5, label="PINN")
axes[0].set_xlabel("t  [s]"); axes[0].set_ylabel("hottest point  [°C]"); axes[0].legend(frameon=False)
axes[1].semilogy(TAUS[1:] * pb.T_END, e_fdm[1:], label="finite differences, 321 x 321")
axes[1].semilogy(TAUS[1:] * pb.T_END, e_pin[1:], label="PINN")
axes[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 0.20 K")
axes[1].set_xlabel("t  [s]"); axes[1].set_ylabel("worst error over the plate  [K]"); axes[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex08.2_report.png"), dpi=150); plt.show()

**What you should see.** The three curves of the hottest point on top of each
other, rising to about 49.4 °C. Finite differences are within 1.25e-01 K of the
reference and the network within about 1.13e-01 K - both inside the agreed
0.20 K. Finite differences take five to fifteen seconds for the whole window and no
training; the network needs about two minutes of training on a CPU, and then
gives the 41 instants in about a third of a second.

---

## 6 · The inverse problem: the diffusivity from four thermocouples

Now suppose the plate's heat capacity is **not known**, so neither is its
diffusivity, but four thermocouples recorded the warm-up every 2 s, with
0.05 K of noise. Can the same network find $\alpha$?

It is L8.2's **Inverse Diffusivity** slide. One number is trained beside the
weights: $C = e^{\beta}$, an exponential so that it cannot go negative. The
loss gains one term, the misfit to the readings, and the residual is the one
you wrote, now with the trainable $C$. $\beta$ starts at $C$ = 0.30, which is
45 % too low. Nothing else changes - which is the point.

In [ ]:
tau_s, readings = pb.sensor_readings(ref)                     # 41 instants x 4 thermocouples, K above the coolant, with noise
S_pts = to_tensor(np.array([[x, y, tau] for tau in tau_s for x, y in pb.SENSORS]))   # where and when they were read
S_val = to_tensor(readings.reshape(-1, 1) / pb.DELTA_T)       # the readings, scaled

class Inverse(torch.nn.Module):                               # the plate's network and one more trainable number
    def __init__(self):
        super().__init__()
        self.plate = Plate()                                  # a fresh network, the same trial function
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(0.30))))   # C = exp(beta), starts 45 % too low
inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 1000.0                                                  # the weight of the readings
def inverse_loss():
    c = torch.exp(inv.beta)                                   # the diffusivity being identified, always positive
    data = (inv.plate(S_pts) - S_val).pow(2).mean()           # the misfit at the thermocouples
    return residual(inv.plate, pts, c).pow(2).mean() + W_N * edge_loss(inv.plate) + W_D * data
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
C_found = float(torch.exp(inv.beta))
alpha_found = C_found * pb.L_PLATE ** 2 / pb.T_END            # back to m^2/s
print(f"diffusivity: found {alpha_found:.2e} m^2/s, true {pb.ALPHA:.2e} m^2/s, "
      f"off by {100 * (alpha_found / pb.ALPHA - 1):+.2f} %   ({inv_time:.0f} s)")
print(f"heat capacity that implies: {pb.K_PLATE / alpha_found / 1e6:.2f} MJ/(m^3 K), true {pb.RHO_CP / 1e6:.2f}")

**What you should see.** The diffusivity found within about 1 % of the
true 6.87e-05 m²/s, in about two and a half minutes on a CPU - from four thermocouples and a
start that was 45 % off.

---

## 7 · What the notebook says

* **Both methods meet the agreed 0.20 K**, and they agree on what an engineer
  wants: the plate is within 2 % of its steady temperature after 80 s, and
  half-way there after about 13 s.
* **For the forward problem, finite differences win on time.** They give the
  whole window in seconds with no training. The network needs minutes, and its
  only advantage is the one of Ex_08.1: the channel is exact, not a staircase.
* **The start was never a problem**, because it is built into the trial
  function. A start left as a loss term is the one instant the network fits
  last; that is the causality of L8.2.
* **The inverse problem is where the network earns its training.** The same
  code, with one more trainable number and one more loss term, returned the
  diffusivity. A grid solver would have to be run again and again inside an
  optimiser to do that.
* **Only the warm-up identifies the diffusivity.** Once the plate is steady the
  readings no longer change, and they say nothing about how fast it got there.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex08.2_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L8.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "What does switching the heat on add to the steady problem of Ex_08.1, and what did the time L²/α = 145.51 s tell you about the window before anything was solved? (-> L8.2 Q1)": """
""",
    "The trial function is exactly zero at the start. Which of the lecture's remedies for causality is that, and where in time was the network's error largest? (-> L8.2 Q2)": """
""",
    "How was the transient checked when it has no exact solution, and why did finite differences need 321 × 321 nodes for 0.20 K? (-> L8.2 Q3)": """
""",
    "In the inverse problem, what was trained besides the weights, why as an exponential, and how close did four thermocouples bring it to the true diffusivity? (-> L8.2 Q4)": """
""",
    "To conclude: finite differences gave the whole window in seconds. For which task in this notebook did the network do something the grid solver could not do directly? (-> L8.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | hottest point at 80 s | worst error | whole window | training |", "|---|---|---|---|---|",
            f"| reference, fitted mesh L/200, 320 steps | {pb.T_COOLANT + hot[-1]:.2f} °C | - | - | - |",
            f"| finite differences, 321 x 321, 320 steps | {pb.T_COOLANT + pb.kelvin(U[-1].max()):.2f} °C | {e_fdm.max():.2e} K | {t_fdm:.2f} s | - |",
            f"| PINN, 4 x 32 | {pb.T_COOLANT + pb.kelvin(P_all[-1].max()):.2f} °C | {e_pin.max():.2e} K | {t_pin:.2f} s | {train_time:.0f} s |",
            "", f"Inverse problem: diffusivity found {alpha_found:.2e} m^2/s, true {pb.ALPHA:.2e} m^2/s, "
                f"off by {100 * (alpha_found / pb.ALPHA - 1):+.2f} %, in {inv_time:.0f} s"]
    out = ["# Ex_08.2 — The Plate Switched On", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex08.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.2_report.md into Ex08.2_report.pdf, with any figure
# saved as Ex08.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex08.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex08.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP8.2A · A moving heat source

**The problem.** The plate of this set was heated evenly. Here a small, intense source moves
across it - a laser, a welding torch, a cutting tool - leaving a hot trail
that cools behind it. The temperature has a sharp peak that travels, and most
of the plate is quiet at any instant.

| | |
|---|---|
| plate | 80 × 40 mm steel, 2 mm thick, 45 W/m·K, 3.6 MJ/m³·K, edges held at 20 °C |
| source | 40 W absorbed, a Gaussian spot of 1 mm, along $y$ = 20 mm from $x$ = 10 to 70 mm at 10 mm/s |
| window | 0 to 8 s: 6 s of heating, 2 s of cooling |

Steel's properties and the absorbed power are typical values, sourced before
the data is released.

**Why a plain PINN is not enough.** Points spread evenly in space and time almost never land on the peak, and a
network that misses the peak misses the only number that matters: how hot the
material got, and for how long.

**What you build.** A PINN for the rise $\theta(x, y, t)$ with the source path as a known input,
and a sampling that follows the source: dense near its current position,
sparse elsewhere. You decide how to move the points during training.

**The ground truth you get.** `MP8.2A_truth.npz`, from `tools/miniprojects/ex082_truth.py`: the plate by
cell-centred finite volumes and backward Euler; the matrix never changes, so it
is factorised once. Cells of 0.1 mm, steps of 5 ms, the field every 0.1 s.

**Worked example.** 

| cells | time step | peak rise | when, where | one run |
|---|---|---|---|---|
| 0.4 mm | 20 ms | 89.06 K | 5.96 s, (68.6, 19.8) mm | 0.6 s |
| 0.2 mm | 10 ms | 89.59 K | 6.00 s, (69.1, 19.9) mm | 10.4 s |
| 0.1 mm | 5 ms | 89.81 K | 6.00 s, (69.2, 20.0) mm | 108.2 s |

The peak comes at the end of the path, where the plate ahead of the spot has
already been warmed by the trail. Two seconds later the plate is still 23.28 K
up.

**Requirements.**

* The peak rise along the path within **2 K** at every instant of the heating.
* The temperature within **3 K** everywhere, at every recorded instant.
* The whole 8 s on the 0.1 mm grid in under **5 s** - against 108 s for the finite volumes.

**What you hand in.** The error in space and time against the ground truth, the peak along the path
against the true one, how much the sampling that follows the source bought you
over uniform points, and the times.

### MP8.2B · Finding a hidden flaw by thermography

**The problem.** Section 6 of this set found one diffusivity for the whole plate. A delamination inside a
composite panel holds heat back only where it is. Flash thermography - the
standard non-destructive test - heats the surface with a flash and watches it
cool with an infrared camera: over a flaw the surface stays warmer, for a
while. The question is where the flaw is, how deep, and how large.

| | |
|---|---|
| section | a 40 mm wide cut through a 4 mm laminate, insulated all round |
| laminate | diffusivity 2e-06 m²/s in-plane and 4e-07 m²/s through the thickness |
| flash | 10 kJ/m² into the top 0.1 mm at $t$ = 0 |
| camera | the surface temperature, every 0.02 s for 10 s, across the 40 mm |
| case 1 | a delamination 10 mm wide at 1 mm deep, 0.2 mm thick, its diffusivity a tenth; noise 0.02 K |
| case 2 | 5 mm wide at 1 mm deep; noise 0.02 K |
| case 3 | 5 mm wide at 2 mm deep; noise 0.05 K |

**Why a plain PINN is not enough.** A diffusivity field is far more to recover than section 6's single number,
and the flaw shows at the surface only in a short window after the flash - and
the deeper it is, the later, the weaker and the more spread out.

**What you build.** An inverse PINN: one network for $\theta(x, z, t)$ and one for the
through-thickness diffusivity $\alpha(x, z)$, kept positive, fitted to the
heat equation, the insulated walls, the flash and the camera frames.

**The ground truth you get.** `MP8.2B_readings.npz` to fit and `MP8.2B_truth.npz` to score against,
from the same script: finite volumes with the conductivity set cell by cell,
cells of 0.1 × 0.025 mm and steps of 5 ms.

**Worked example.** Case 1: the surface over the flaw runs up to **1.14 K** warmer than an
undamaged laminate, 2.86 s after the flash, at its centre, $x$ = 20 mm. The
contrast is 1.143, 1.143 and 1.144 K on the three grids - fifty times the
noise.

**Requirements.**

* Case 1: the flaw's edges within **1 mm** and its depth within **0.2 mm**.
* Every case: say whether there is a flaw, and where; case 3 within **2 mm**.
* The surface temperature within **0.05 K** over the 10 s.

**What you hand in.** The recovered diffusivity against the true one in each case, where your network
puts each flaw, and the smallest and deepest flaw it still finds at the given
noise.